In [1]:
import os
import numpy as np
import librosa
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
import joblib

In [2]:
# =========================
# CONFIG
# =========================
data_dir = r"D:\Basant\Graduation Project\AI\AI-online\Final Data\BALANCED_DATA"
emergency_classes = ["Siren", "Gunshot", "Cracking"]

SAMPLE_RATE = 16000
DURATION = 3
TARGET_LEN = SAMPLE_RATE * DURATION  # عدد العينات بالضبط

In [3]:
# =========================
# LOAD RAW AUDIO (مفيش features، صوت خام بس)
# =========================
def load_raw_audio(file_path):
    audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)

    # توحيد الطول (padding أو قطع)
    if len(audio) < TARGET_LEN:
        audio = np.pad(audio, (0, TARGET_LEN - len(audio)))
    else:
        audio = audio[:TARGET_LEN]

    # توحيد الصوت (normalize بسيط، بدون مكتبات خاصة)
    max_val = np.max(np.abs(audio))
    if max_val == 0:
        max_val = 1e-9
    audio = audio / max_val

    return audio.astype(np.float32)


In [4]:
# =========================================================
# PART 1: EMERGENCY MODEL (Normal vs Emergency)
# =========================================================
print("=" * 50)
print("Loading data for EMERGENCY model...")
print("=" * 50)

X_emergency, y_emergency = [], []

for cls in os.listdir(data_dir):
    class_path = os.path.join(data_dir, cls)
    if not os.path.isdir(class_path):
        continue

    for file in tqdm(os.listdir(class_path), desc=cls):
        path = os.path.join(class_path, file)
        try:
            audio = load_raw_audio(path)
            X_emergency.append(audio)
            y_emergency.append(1 if cls in emergency_classes else 0)
        except Exception:
            pass

X_emergency = np.array(X_emergency)
y_emergency = np.array(y_emergency)

print("Dataset shape:", X_emergency.shape)
print("Emergency:", sum(y_emergency), "Normal:", len(y_emergency) - sum(y_emergency))

X_emergency, y_emergency = shuffle(X_emergency, y_emergency, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X_emergency, y_emergency,
    test_size=0.2,
    random_state=42,
    stratify=y_emergency
)

# Reshape لـ Conv1D: (samples, time_steps, channels)
X_train = X_train.reshape(-1, TARGET_LEN, 1)
X_test = X_test.reshape(-1, TARGET_LEN, 1)


Loading data for EMERGENCY model...


Car:   0%|          | 0/1000 [00:00<?, ?it/s]c:\Users\LEGEND\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Train: 100%|██████████| 1000/1000 [00:00<00:00, 1082.86it/s]


Dataset shape: (10000, 48000)
Emergency: 3000 Normal: 7000


In [5]:
# =========================
# EMERGENCY MODEL (Conv1D على الصوت الخام)
# =========================
emergency_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(TARGET_LEN, 1)),

    tf.keras.layers.Conv1D(16, kernel_size=64, strides=8, activation='relu'),
    tf.keras.layers.MaxPooling1D(4),

    tf.keras.layers.Conv1D(32, kernel_size=32, strides=4, activation='relu'),
    tf.keras.layers.MaxPooling1D(4),

    tf.keras.layers.Conv1D(64, kernel_size=16, strides=2, activation='relu'),
    tf.keras.layers.GlobalAveragePooling1D(),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

emergency_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

print("\nTraining EMERGENCY model...")
emergency_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate
y_pred_proba = emergency_model.predict(X_test).flatten()

print("\n" + "=" * 50)
print("EMERGENCY MODEL - THRESHOLD EXPERIMENT")
print("=" * 50)

for t in [0.5, 0.4, 0.3, 0.25, 0.2]:
    y_pred_temp = (y_pred_proba > t).astype(int)
    acc_temp = accuracy_score(y_test, y_pred_temp)
    print(f"\nThreshold = {t} | Accuracy = {acc_temp}")
    print(classification_report(y_test, y_pred_temp, target_names=["Normal", "Emergency"]))

os.makedirs("artifacts", exist_ok=True)
emergency_model.save("artifacts/emergency_model_raw.h5")
print("✅ Emergency model (raw audio) saved")



Training EMERGENCY model...
Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 11s 44ms/step - accuracy: 0.7138 - loss: 0.5601 - val_accuracy: 0.7287 - val_loss: 0.5239
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - accuracy: 0.7489 - loss: 0.5026 - val_accuracy: 0.7469 - val_loss: 0.4938
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 40ms/step - accuracy: 0.7791 - loss: 0.4712 - val_accuracy: 0.7731 - val_loss: 0.4400
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - accuracy: 0.8070 - loss: 0.4404 - val_accuracy: 0.8181 - val_loss: 0.4155
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 40ms/step - accuracy: 0.8270 - loss: 0.4059 - val_accuracy: 0.8388 - val_loss: 0.3719
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - accuracy: 0.8405 - loss: 0.3813 - val_accuracy: 0.8344 - val_loss: 0.3690
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - accuracy: 0.8589 - loss: 0.3540 - val_accuracy: 0.8562 - val_loss: 0.3314
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 40ms/step - accu


EMERGENCY MODEL - THRESHOLD EXPERIMENT

Threshold = 0.5 | Accuracy = 0.93
              precision    recall  f1-score   support

      Normal       0.95      0.95      0.95      1400
   Emergency       0.88      0.89      0.88       600

    accuracy                           0.93      2000
   macro avg       0.91      0.92      0.92      2000
weighted avg       0.93      0.93      0.93      2000


Threshold = 0.4 | Accuracy = 0.922
              precision    recall  f1-score   support

      Normal       0.96      0.93      0.94      1400
   Emergency       0.85      0.90      0.87       600

    accuracy                           0.92      2000
   macro avg       0.90      0.92      0.91      2000
weighted avg       0.92      0.92      0.92      2000


Threshold = 0.3 | Accuracy = 0.915
              precision    recall  f1-score   support

      Normal       0.96      0.92      0.94      1400
   Emergency       0.83      0.91      0.87       600

    accuracy                       

In [6]:
# =========================================================
# PART 2: TYPE MODEL (Siren vs Gunshot vs Cracking)
# =========================================================
print("\n" + "=" * 50)
print("Loading data for TYPE model...")
print("=" * 50)

X_type, y_type = [], []

for cls in emergency_classes:
    class_path = os.path.join(data_dir, cls)
    if not os.path.isdir(class_path):
        continue

    for file in tqdm(os.listdir(class_path), desc=cls):
        path = os.path.join(class_path, file)
        try:
            audio = load_raw_audio(path)
            X_type.append(audio)
            y_type.append(cls)
        except Exception:
            pass

X_type = np.array(X_type)
y_type = np.array(y_type)

print("Dataset shape:", X_type.shape)

le = LabelEncoder()
y_type_encoded = le.fit_transform(y_type)
print("Classes:", le.classes_)

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_type, y_type_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_type_encoded
)

X_train2 = X_train2.reshape(-1, TARGET_LEN, 1)
X_test2 = X_test2.reshape(-1, TARGET_LEN, 1)


Loading data for TYPE model...


Cracking: 100%|██████████| 1000/1000 [00:00<00:00, 1099.51it/s]


Dataset shape: (3000, 48000)
Classes: ['Cracking' 'Gunshot' 'Siren']


In [7]:
# =========================
# TYPE MODEL (Conv1D على الصوت الخام)
# =========================
type_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(TARGET_LEN, 1)),

    tf.keras.layers.Conv1D(16, kernel_size=64, strides=8, activation='relu'),
    tf.keras.layers.MaxPooling1D(4),

    tf.keras.layers.Conv1D(32, kernel_size=32, strides=4, activation='relu'),
    tf.keras.layers.MaxPooling1D(4),

    tf.keras.layers.Conv1D(64, kernel_size=16, strides=2, activation='relu'),
    tf.keras.layers.GlobalAveragePooling1D(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

type_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop2 = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

print("\nTraining TYPE model...")
type_model.fit(
    X_train2, y_train2,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop2],
    verbose=1
)

# Evaluate
y_pred_prob2 = type_model.predict(X_test2)
y_pred2 = np.argmax(y_pred_prob2, axis=1)

print("\n📊 TYPE MODEL Classification Report:\n")
print(classification_report(y_test2, y_pred2, target_names=le.classes_))
print("\n📌 Confusion Matrix:\n")
print(confusion_matrix(y_test2, y_pred2))


Training TYPE model...
Epoch 1/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - accuracy: 0.5786 - loss: 0.8223 - val_accuracy: 0.6438 - val_loss: 0.6049
Epoch 2/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 42ms/step - accuracy: 0.6661 - loss: 0.6241 - val_accuracy: 0.7042 - val_loss: 0.5576
Epoch 3/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - accuracy: 0.6938 - loss: 0.6043 - val_accuracy: 0.7563 - val_loss: 0.5087
Epoch 4/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.7885 - loss: 0.5161 - val_accuracy: 0.8188 - val_loss: 0.4309
Epoch 5/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.8151 - loss: 0.4538 - val_accuracy: 0.8396 - val_loss: 0.3998
Epoch 6/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.8500 - loss: 0.3887 - val_accuracy: 0.8479 - val_loss: 0.3793
Epoch 7/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.8510 - loss: 0.3718 - val_accuracy: 0.8708 - val_loss: 0.3827
Epoch 8/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.8547 - loss: 0

In [8]:
type_model.save("artifacts/type_model_raw.h5")
joblib.dump(le, "artifacts/type_encoder_raw.pkl")
print("✅ Type model (raw audio) saved")

print("\n" + "=" * 50)
print("DONE ✅ BOTH MODELS READY FOR TFLITE CONVERSION 🚀")
print("=" * 50)

✅ Type model (raw audio) saved

DONE ✅ BOTH MODELS READY FOR TFLITE CONVERSION 🚀
